# DecodeLabs Project 2
## Supervised Learning - Fraud Detection Pipeline

Complete notebook with SMOTE, Logistic Regression, Random Forest, GridSearchCV and evaluation metrics.

In [ ]:
!pip -q install imbalanced-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,ConfusionMatrixDisplay,RocCurveDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [ ]:
uploaded=files.upload()
df=pd.read_csv(list(uploaded.keys())[0])
df.head()

In [ ]:
df['Date']=pd.to_datetime(df['Date'])
df['Year']=df['Date'].dt.year
df['Month']=df['Date'].dt.month
df['DayOfWeek']=df['Date'].dt.dayofweek
q=df.Quantity.quantile(.95);p=df.TotalPrice.quantile(.95);c=df.ItemsInCart.quantile(.95)
df['Fraud_Risk_Score']=((df.Quantity>q).astype(int)+(df.TotalPrice>p).astype(int)+(df.ItemsInCart>c).astype(int))
df['IsFraud']=(df.Fraud_Risk_Score>=2).astype(int)

In [ ]:
features=['Quantity','UnitPrice','ItemsInCart','TotalPrice','PaymentMethod','OrderStatus','CouponCode','ReferralSource','Year','Month','DayOfWeek']
X=df[features];y=df['IsFraud']
num=['Quantity','UnitPrice','ItemsInCart','TotalPrice','Year','Month','DayOfWeek']
cat=['PaymentMethod','OrderStatus','CouponCode','ReferralSource']
pre=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

In [ ]:
lr=Pipeline([('pre',pre),('smote',SMOTE(random_state=42)),('clf',LogisticRegression(max_iter=1000))])
rf=Pipeline([('pre',pre),('smote',SMOTE(random_state=42)),('clf',RandomForestClassifier(random_state=42))])
glr=GridSearchCV(lr,{'clf__C':[0.1,1,10]},cv=5,scoring='roc_auc')
grf=GridSearchCV(rf,{'clf__n_estimators':[100,200],'clf__max_depth':[None,10]},cv=5,scoring='roc_auc')
glr.fit(X_train,y_train);grf.fit(X_train,y_train)

In [ ]:
models={'Logistic Regression':glr.best_estimator_,'Random Forest':grf.best_estimator_}
rows=[]
for n,m in models.items():
 p=m.predict(X_test);pr=m.predict_proba(X_test)[:,1]
 rows.append([n,accuracy_score(y_test,p),precision_score(y_test,p,zero_division=0),recall_score(y_test,p,zero_division=0),f1_score(y_test,p,zero_division=0),roc_auc_score(y_test,pr)])
results=pd.DataFrame(rows,columns=['Model','Accuracy','Precision','Recall','F1','ROC-AUC'])
results

In [ ]:
best=models[results.sort_values('ROC-AUC',ascending=False).iloc[0]['Model']]
ConfusionMatrixDisplay.from_predictions(y_test,best.predict(X_test));plt.show()
RocCurveDisplay.from_estimator(best,X_test,y_test);plt.show()
results.to_csv('Project2_Model_Comparison.csv',index=False)
df.to_csv('Project2_Dataset_With_Fraud_Label.csv',index=False)
print('Completed')